# 4 · Plantilla para cualquier conjunto de datos

El mismo procedimiento sirve para **cualquier** fuente que Node-RED pueda leer. Cambia solo la configuración
de la primera celda y ejecuta todo (**Run → Run All Cells**).

Por defecto está preparada para **SolarLab** (flujo *Serie 4 · SolarLab*): potencia esperada a partir de la
irradiancia y la temperatura del panel, y anomalías.

In [ ]:
CONJUNTO = "solarlab"                       # datos/<CONJUNTO>.csv
OBJETIVO = "potencia"                       # lo que se quiere predecir
ENTRADAS = ["irradiancia", "temp_panel"]    # con qué se predice
ETIQUETA = "averia"                         # columna con la verdad (o None si no hay)
NORMAL = 0                                  # valor de la etiqueta en funcionamiento normal

In [ ]:
from labia import datos, guardar
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = datos(CONJUNTO).dropna(subset=ENTRADAS + [OBJETIVO])
normal = df if ETIQUETA is None else df[df[ETIQUETA] == NORMAL]
print(len(df), "filas;", len(normal), "normales")
if ETIQUETA:
    print(df[ETIQUETA].value_counts())

In [ ]:
pd_ = df.copy()
fig, ejes = plt.subplots(1, len(ENTRADAS), figsize=(5 * len(ENTRADAS), 4), squeeze=False)
for ax, col in zip(ejes[0], ENTRADAS):
    grupos = pd_.groupby(ETIQUETA) if ETIQUETA else [("todos", pd_)]
    for et, g in grupos:
        ax.scatter(g[col], g[OBJETIVO], s=5, alpha=.4, label=str(et))
    ax.set_xlabel(col); ax.set_ylabel(OBJETIVO); ax.grid(alpha=.3); ax.legend(title=ETIQUETA)

## Modelo de regresión (red neuronal)

In [ ]:
Xe, Xp, ye, yp = train_test_split(normal[ENTRADAS], normal[OBJETIVO], test_size=0.25, random_state=0)
red = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 32), max_iter=3000, random_state=0))
red.fit(Xe, ye)
mae = mean_absolute_error(yp, red.predict(Xp))
print(f"MAE = {mae:.1f}   R² = {r2_score(yp, red.predict(Xp)):.3f}")
df["desviacion"] = df[OBJETIVO] - red.predict(df[ENTRADAS])
if ETIQUETA:
    df.boxplot(column="desviacion", by=ETIQUETA, figsize=(8, 4)); plt.suptitle("")

In [ ]:
guardar(red, f"{CONJUNTO}-curva", entradas=ENTRADAS, tipo="regresion", salida=OBJETIVO,
        metricas={"mae": round(mae, 2)})

## Detector de anomalías

In [ ]:
CAMPOS = ENTRADAS + [OBJETIVO]
iso = make_pipeline(StandardScaler(), IsolationForest(contamination=0.01, random_state=0)).fit(normal[CAMPOS])
df["anomalia"] = iso.predict(df[CAMPOS]) == -1
if ETIQUETA:
    print((df.groupby(ETIQUETA)["anomalia"].mean() * 100).round(1).to_frame("% marcadas como anómalas"))
guardar(iso, f"{CONJUNTO}-anomalias", entradas=CAMPOS, tipo="anomalias")

## Comparar: ¿quién detecta mejor?

Otra forma de detectar problemas es usar el **modelo de regresión**: el *rendimiento* = real / esperado.
Si la máquina produce menos de un 90 % de lo que el modelo espera, algo pasa.

In [ ]:
esperada = red.predict(df[ENTRADAS])
df["rendimiento"] = df[OBJETIVO] / np.clip(esperada, abs(esperada).max() * 0.02, None)
df["bajo"] = df["rendimiento"] < 0.9
if ETIQUETA:
    print(pd.DataFrame({"rendimiento medio": df.groupby(ETIQUETA)["rendimiento"].mean().round(2),
                        "% rendimiento < 0,9": (df.groupby(ETIQUETA)["bajo"].mean() * 100).round(1),
                        "% anomalía (Isolation Forest)": (df.groupby(ETIQUETA)["anomalia"].mean() * 100).round(1)}))

## ✏️ Tu turno

1. ¿Por qué el detector de anomalías sobre los datos **en bruto** puede no ver una avería que la curva sí ve?
   (pista: una potencia de 2000 kW es *habitual*… con otra irradiancia).
2. Entrena el Isolation Forest con el **rendimiento** como entrada (en Node-RED tendrías que calcularlo antes de
   **ia predecir**). ¿Mejora?
3. Repite el cuaderno con **otro conjunto** que recojas tú con **ia registrar** (OceanLab, AtomLab, ReactorLab…):
solo cambian las cinco variables de la primera celda. ¿Qué predecirías en cada simulador?